# 详细文档: Preissmann模型 - 数值求解

**相关模块:**
- `preissmann_model/model.py`
- `preissmann_model/solver.py`

## 目的
此文档旨在深入探讨`preissmann_model`的核心——用于求解圣维南方程组的数值方法。模型采用了经典的**Preissmann四点加权隐式差分格式**。此方法将非线性的偏微分方程（圣维南方程）在时间和空间上离散化和线性化，最终转化为一个大型的线性方程组，然后进行求解。

此笔记本将：
1.  简要介绍Preissmann格式的基本思想。
2.  剖析`HydraulicModel.step()`方法，展示它如何为整个河段构建一个大型的线性方程组 `M * dX = R`。
3.  可视化这个大型矩阵`M`的稀疏结构，以更直观地理解方程组的全貌。
4.  指出`solver.py`中的分块三对角矩阵求解器目前并未被`model.py`使用，后者采用了`numpy.linalg.solve`直接求解整个大矩阵。

## 1. Preissmann格式简介

圣维南方程组由连续性方程和动量方程组成，它们都是非线性的偏微分方程，难以求得解析解。Preissmann格式通过以下方式将其转化为可解的线性方程：

- **空间离散**: 将河段离散为一系列计算节点（`i`）和它们之间的河段（`i`到`i+1`）。
- **时间离散**: 将时间离散为`n`和`n+1`两个时刻。
- **四点加权**: 所有的变量（如`Q`, `Z`）和系数都在`i`, `i+1`, `n`, `n+1`这四个点上进行加权平均。
- **线性化**: 使用泰勒级数展开，将方程在`n`时刻的已知值附近进行线性化，得到一个关于未知增量 `dZ` 和 `dQ` 的线性方程。

对于每一个小河段（`i`到`i+1`），这个过程会产生两个线性方程，包含四个未知数（`dZ_i`, `dQ_i`, `dZ_{i+1}`, `dQ_{i+1}`）。将所有河段的方程、边界条件方程和内部结构物（如闸门、水泵）的方程组合在一起，就形成了一个大型的、但通常是稀疏的线性方程组 `M * dX = R`。

## 2. 构建方程组: `HydraulicModel.step()` 剖析

`HydraulicModel.step()`方法是模型的心脏。在每个时间步，它执行以下操作：
1.  **初始化**: 创建一个全零的大矩阵 `M` 和右端项向量 `R`。
2.  **组装内部方程**: 循环遍历每个河段（`i`到`i+1`），调用 `_get_segment_equations(i)` 方法。该方法根据当前时刻的水力条件，计算出该河段的连续性方程和动量方程的线性化系数，并将这些系数填充到`M`矩阵的相应位置。
3.  **施加上游边界条件**: 修改`M`和`R`的第一行（或与第一个节点相关的行），以反映上游的入流（`Q_inflow`）或水位（`Z_inflow`）边界条件。
4.  **施加下游边界条件**: 修改`M`和`R`的最后一行，以反映下游的水位边界条件。
5.  **施加内部结构物方程**: 如果存在闸门或水泵等结构物，用它们的物理方程（也是线性化的）替换掉它们所在位置的动量方程。
6.  **求解**: 调用`numpy.linalg.solve(M, R)`求解增量向量`dX`。
7.  **更新状态**: 将求解出的增量`dX`加回到当前状态`Z`和`Q`上，得到下一个时间步的状态。

下面，我们用一个例子来实际构建并观察这个矩阵`M`。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys
import os

# 将项目根目录添加到Python路径
sys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname('__file__'), '..')))

from preissmann_model.model import HydraulicModel
from preissmann_model.reach import RiverReach
from preissmann_model.cross_section import RectangularCrossSection
from preissmann_model.structures import Gate

# --- 创建一个带闸门的简单模型，用于演示 ---
num_nodes = 11
reach_geom = RiverReach(
    cross_sections=[RectangularCrossSection(width=10) for _ in range(num_nodes)],
    lengths=np.full(num_nodes - 1, 250.0),
    slope=0.001,
    manning_n=0.03
)
gate = Gate(name="test_gate", node_index=5, opening_height=0.5, width=10)

river = HydraulicModel(
    name="R1_for_solver_demo",
    reach=reach_geom,
    dt=10.0,
    downstream_level=2.0,
    structures=[gate]
)

# 设置初始条件
river.Q[:] = 10.0
for i in range(num_nodes):
    river.Z[i] = river.Z_bed[i] + 2.2

## 3. 可视化系数矩阵 `M`

现在，我们手动调用`step`方法中的代码来构建矩阵`M`，然后使用`matplotlib.pyplot.spy`来可视化它的稀疏结构。

In [ ]:
def build_matrix_for_inspection(model, inflows):
    n_vars = model.num_nodes * 2
    M = np.zeros((n_vars, n_vars))
    
    # 内部方程
    for i in range(model.num_nodes - 1):
        aa_i, bb_i, _ = model._get_segment_equations(i)
        row = i * 2
        M[row:row+2, 2*i:2*i+2] = aa_i
        M[row:row+2, 2*(i+1):2*(i+1)+2] = bb_i
        
    # 上游边界
    M[n_vars-2, :] = 0
    M[n_vars-2, 1] = 1.0 # dQ_0 = ...
    
    # 下游边界
    M[n_vars-1, :] = 0
    M[n_vars-1, n_vars-2] = 1.0 # dZ_end = ...
    
    # 闸门
    s = model.structure_map[5]
    i = s.node_index
    row_to_replace = (i-1)*2 + 1
    M[row_to_replace, :] = 0
    coeffs, _ = s.get_linearized_equation(model.Z[i-1], model.Z[i], model.Q[i], model.g)
    M[row_to_replace, (i-1)*2] = coeffs.get('dZ_up', 0.0)
    M[row_to_replace, i*2] = coeffs.get('dZ_down', 0.0)
    M[row_to_replace, i*2 + 1] = coeffs.get('dQ', 0.0)
    
    return M

M_matrix = build_matrix_for_inspection(river, {'Q_inflow': 10.0})

fig, ax = plt.subplots(figsize=(10, 10))
ax.spy(M_matrix, markersize=5)
ax.set_title('Structure of the Coefficient Matrix M')
ax.set_xlabel('Column index (Variable)')
ax.set_ylabel('Row index (Equation)')
plt.show()

### 矩阵结构解读

上图中的每个点代表矩阵`M`中的一个非零元素。
- **主体结构**: 大部分非零点集中在对角线附近，形成一个“带状”结构。这是因为每个河段的方程只关联相邻两个节点（`i`和`i+1`）的变量，这体现了所谓的分块三对角（block-tridiagonal）特性。
- **边界条件**: 右上角和右下角的两个单独的点分别对应上下游边界条件的方程行。我们看到它们被修改为只有单个非零元素（表示`1 * dQ_0 = ...` 或 `1 * dZ_end = ...`）。
- **闸门方程**: 在矩阵中间，有一行（`row=9`）的模式与其他行不同。它有三个非零元素，对应闸门方程中的`dZ_up`, `dZ_down`, `dQ`三个变量。这一行替换了原来的动量方程行。

**关于`solver.py`的说明:**
虽然这种带状矩阵最高效的求解方法是分块三对角矩阵算法（如 `solver.py` 中实现的 `solve_block_tridiagonal`），但当前的`model.py`为了实现的简便性，直接构建了整个稀疏矩阵`M`，并使用`numpy.linalg.solve`进行求解。对于节点数不多的情况，`numpy`的求解器已经足够高效。如果未来需要模拟非常长（数千个节点）的河流，切换到专门的稀疏矩阵求解器或`solve_block_tridiagonal`算法会是重要的性能优化方向。